# Nível do modo residual com e sem os pesos das pontas exatos, numa T4

Antes de rodar: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois *Executar tudo*.

Os níveis 1 e 2 do modo residual guardam exatos os pesos que caem fora da grade do Q4_K (0,14%). Um GGUF Q5_K não tem onde guardá-los e os corta na ponta. Este notebook mede, no wikitext-2 de teste inteiro, cada nível das duas formas, mais Q5_K_M e Q6_K na mesma rodada. A pergunta: a vantagem do nível sobre o k-quant vem desses pesos? Se sim, o kernel precisa de uma lista à parte para eles. Leva uns 45 minutos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q -U transformers datasets safetensors
!pip install -q -e llama.cpp/gguf-py
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=OFF > /dev/null && cmake --build build -j 2 --target llama-quantize 2>&1 | grep -E 'Linking|error' ; ls build/bin

Converter para GGUF BF16 e quantizar, sem imatrix (o `llama-quantize` padrão).

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download("unsloth/Llama-3.2-1B", allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
!mkdir -p /content/gguf && python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile /content/gguf/bf16.gguf 2>&1 | tail -1
for q in ["Q4_K_M", "Q5_K_M", "Q6_K"]:
    !llama.cpp/build/bin/llama-quantize /content/gguf/bf16.gguf /content/gguf/{q}.gguf {q} > /content/gguf/{q}.log 2>&1
!ls -la /content/gguf

In [ ]:
%cd /content/strata
!python tests/ppl/strata_residual.py --ppl --model unsloth/Llama-3.2-1B --gguf Q4_K_M=/content/gguf/Q4_K_M.gguf --niveis 1,2 --sem-saturados --ref Q5_K_M=/content/gguf/Q5_K_M.gguf,Q6_K=/content/gguf/Q6_K.gguf --out /content/res
!cd /content/res && cp unsloth__Llama-3.2-1B_residual_ppl.md unsloth__Llama-3.2-1B_saturados.md && cp unsloth__Llama-3.2-1B_residual_ppl.json unsloth__Llama-3.2-1B_saturados.json
from IPython.display import Markdown, display
display(Markdown(open('/content/res/unsloth__Llama-3.2-1B_saturados.md').read()))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/unsloth__Llama-3.2-1B_saturados.md')
files.download('/content/res/unsloth__Llama-3.2-1B_saturados.json')